In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import stats

DATA_DIR = Path("../data/experiment_results_raw")

METRICS = {
    "Process energy per run": "proc_energy_per_run_J",
    "System energy per run (net)": "net_sys_host_per_run_J",
}

def remove_outliers(x, lower_q=0.005, upper_q=0.995):
    lo, hi = np.quantile(x, [lower_q, upper_q])
    return x[(x >= lo) & (x <= hi)]

experiments = {}
for f in DATA_DIR.glob("energy_results_*.json"):
    stem = f.stem.replace("energy_results_", "")
    op, suffix = stem.rsplit("_", 1)
    experiments.setdefault(op, {})[suffix] = f

for operation, files in experiments.items():
    if len(files) != 2:
        continue

    frames = []
    for suffix, path in files.items():
        with open(path) as f:
            trials = json.load(f)["trials"]
        df = pd.DataFrame(trials)
        df["variant"] = suffix
        frames.append(df)

    df = pd.concat(frames, ignore_index=True)

    for c in df.columns:
        if c.endswith("_J"):
            df[c.replace("_J", "_mJ")] = df[c] * 1000

    print(f"\nOperation: {operation}")

    for label, base_col in METRICS.items():
        col = base_col.replace("_J", "_mJ")
        print(f"\n{label}")

        for variant in sorted(df["variant"].unique()):
            x = df[df["variant"] == variant][col].dropna().to_numpy()
            x = remove_outliers(x)

            if len(x) < 3:
                print(f"{variant}: n<3")
                continue

            w, p = stats.shapiro(x)
            result = "NORMAL" if p >= 0.05 else "NOT normal"

            print(f"{variant} | n={len(x)} | W={w:.4f} | p={p:.4e} ({result})")


Operation: validate_billing

Process energy per run
optimized | n=18 | W=0.8798 | p=2.5882e-02 (NOT normal)
unoptimized | n=18 | W=0.9178 | p=1.1813e-01 (NORMAL)

System energy per run (net)
optimized | n=18 | W=0.9415 | p=3.0738e-01 (NORMAL)
unoptimized | n=18 | W=0.9252 | p=1.6003e-01 (NORMAL)

Operation: reorder_item

Process energy per run
optimized | n=18 | W=0.9655 | p=7.0986e-01 (NORMAL)
unoptimized | n=18 | W=0.7687 | p=5.6714e-04 (NOT normal)

System energy per run (net)
optimized | n=19 | W=0.8413 | p=4.8544e-03 (NOT normal)
unoptimized | n=19 | W=0.2439 | p=5.2804e-09 (NOT normal)

Operation: generate_report

Process energy per run
optimized | n=18 | W=0.9858 | p=9.9013e-01 (NORMAL)
unoptimized | n=18 | W=0.4192 | p=1.7192e-07 (NOT normal)

System energy per run (net)
optimized | n=18 | W=0.9318 | p=2.0898e-01 (NORMAL)
unoptimized | n=18 | W=0.9736 | p=8.6262e-01 (NORMAL)

Operation: bulk_import

Process energy per run
optimized | n=18 | W=0.8789 | p=2.4973e-02 (NOT normal)